# 04. Mask-Based Image Editing (Inpainting)

This notebook demonstrates **mask-based image editing** (a.k.a. inpainting) using Azure OpenAI's
`images.edit()` endpoint with a `mask` argument. Unlike notebook 03's prompt-only edit, a mask
gives the model a **hard, pixel-precise boundary**: only the masked (transparent) region of
`product_photo.png` is eligible to change — everything else is guaranteed to stay untouched. The
result is saved to `masked_edit_product_photo.png`. This is the third step in the
*generate → prompt-edit → mask-edit* progression (02 → 03 → 04).

**Difficulty:** Intermediate — requires understanding how image masks work, which trips up a lot
of newcomers.

## Prerequisites

**Python packages** (already in the repo's root `requirements.txt` — no extra install needed):
- `openai`, `python-dotenv`
- `base64`, `pathlib` — standard library
- `Pillow` (`PIL`) is **not** imported by this script — it reads `mask.png` as a plain binary
  file and hands it straight to the API. Pillow is already in `requirements.txt` and useful if
  *you* want to programmatically create/edit mask images yourself (see Try It Yourself below),
  but it's not required to run this notebook as-is.

**Azure resources**
- The same Azure OpenAI **image model deployment** used in notebooks 02-03 (e.g. `gpt-image-2`).

**Environment variables** — reuse the same names as notebooks 02-03, added to the root `.env`:
```
AZURE_OPENAI_IMAGE_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
AZURE_OPENAI_IMAGE_API_KEY=<your-api-key>
AZURE_OPENAI_IMAGE_DEPLOYMENT=gpt-image-2
```

**Input files:** `product_photo.png` and `mask.png` must both exist in this folder (they already
do, as sample assets) and must have matching dimensions.

## What You'll Learn

- **What an image mask actually is:** a PNG with an alpha (transparency) channel, where the
  **transparent region (alpha = 0) marks the area the model is allowed to regenerate**, and the
  **opaque region (alpha = 255) marks the area that must be preserved exactly**. This is the
  single most common point of confusion for people new to inpainting APIs — it's easy to assume
  masks work the opposite way (highlighting what to keep) when they actually mark what to
  *replace*.
- Why mask-based editing gives stronger guarantees than prompt-only editing (notebook 03)
- That the mask image must match the source image's dimensions
- How to pass both an `image` and a `mask` file to `images.edit()` at the same time

### Imports and setup

Identical imports to notebook 03 — this script is structurally the same call, with one added
argument (`mask=`).

💡 **Exam tip:** AI-102 material on DALL-E/image-model editing describes masking as "inpainting":
you supply an original image and a mask, and only the masked region is regenerated. Know this
distinguishes *editing* (targeted, constrained) from *generation* (unconstrained, from scratch).

🔄 **Alternatives:** This entire notebook is itself the "alternative" to notebook 03 — mask-based
edit vs. prompt-only edit.

In [ ]:
import os
from pathlib import Path
import base64

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

### Configuration and file paths

- Three paths are defined: the source `product_photo.png`, the `mask.png` that marks the
  editable region, and the `masked_edit_product_photo.png` output destination.
- The client construction is identical to notebooks 02-03.

💡 **Exam tip:** Remember the mask's transparent (alpha = 0) area is what gets **regenerated**;
everything opaque is preserved pixel-for-pixel from the original — this is the opposite of what
many people intuitively guess on first exposure to inpainting.

In [ ]:
endpoint = os.getenv("AZURE_OPENAI_IMAGE_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_IMAGE_API_KEY")
IMAGE_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_IMAGE_DEPLOYMENT", "gpt-image-2")

input_image_path = Path("product_photo.png")
mask_image_path = Path("mask.png")
output_image_path = Path("masked_edit_product_photo.png")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

### The edit prompt

The prompt here describes *only what should appear in the editable area* ("a small modern wall
display showing a simple quarterly sales chart") and instructs the model to match the
surrounding lighting/perspective. It doesn't need to say "keep everything else unchanged" as
forcefully as notebook 03's prompt did — the mask itself is what enforces that boundary.

In [ ]:
prompt = """
In the editable area, add a small modern wall display showing a simple quarterly sales chart.
Keep the rest of the image unchanged.
The style should match the lighting and perspective of the original photo.
"""

### Calling images.edit() with a mask

- Both the source image and the mask are opened in binary mode and passed as separate file
  objects: `image=image_file` and `mask=mask_file`.
- The API uses the mask's alpha channel to determine which pixels of `image` are eligible to be
  regenerated according to `prompt` — the opaque pixels are copied through unchanged.

💡 **Exam tip:** The mask file must be a PNG with the **same dimensions** as the source image;
mismatched dimensions will cause the request to fail.

🔄 **Alternatives:** Related but distinct techniques you may see referenced in Azure OpenAI image
docs: **outpainting** (extending an image's canvas beyond its original borders, effectively
masking the new "empty" surrounding area) uses this same masked-edit mechanism, just with the
mask covering newly-added canvas rather than a region inside the original image.

In [ ]:
with input_image_path.open("rb") as image_file, mask_image_path.open("rb") as mask_file:
    response = client.images.edit(
        model=IMAGE_DEPLOYMENT_NAME,
        image=image_file,
        mask=mask_file,
        prompt=prompt,
        size="1024x1024",
        n=1,
        quality="medium"
    )

### Decoding and saving the result

Same decode-and-write pattern as notebooks 02-03. Running this notebook overwrites
`masked_edit_product_photo.png` in the working directory (the sample output already present in
this folder).

In [ ]:
image_base64 = response.data[0].b64_json
image_bytes = base64.b64decode(image_base64)

output_image_path.write_bytes(image_bytes)

print(f"Masked edited image saved to: {output_image_path}")

## Summary

This notebook performed a **masked edit (inpainting)**: it sent `product_photo.png` alongside
`mask.png` to `images.edit()`, so the model only regenerated the transparent region of the mask
(adding a wall display with a sales chart) while leaving every opaque pixel of the original photo
untouched. The result was saved to `masked_edit_product_photo.png`. Compared to notebook 03's
prompt-only edit, this gives a structural guarantee about what can and can't change, rather than
relying entirely on the model following instructions.

## Try It Yourself

1. **Easy:** Open `mask.png` in an image viewer (or with Pillow: `Image.open("mask.png")`) and
   inspect which region is transparent vs. opaque before running the edit, to confirm your
   mental model of which area will change.
2. **Intermediate:** Use Pillow to create a new mask with a different transparent shape/location
   (e.g. `Image.new("RGBA", size, (0,0,0,255))` then punch a transparent rectangle into it with
   `ImageDraw`), and edit a different region of `product_photo.png`.
3. **Advanced:** Create a mask that is transparent along one entire edge of the image (rather
   than an interior region) to experiment with outpainting-style canvas extension, and compare
   how the model handles blending at the boundary versus an interior edit.